Reconocimiento de Rostros con tensor Flow

In [5]:
import cv2
import os

nombre = "Groversingorra"
ruta = f'dataset_tf/{nombre}'
os.makedirs(ruta, exist_ok=True)

cap = cv2.VideoCapture(0)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                     'haarcascade_frontalface_default.xml')

cont = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray,
                                          scaleFactor=1.1,
                                          minNeighbors=5,
                                          minSize=(30, 30))
    for (x, y, w, h) in faces:
        rostro = gray[y:y+h, x:x+w]
        # redimensionamos para el modelo
        rostro_resized = cv2.resize(rostro, (100, 100))
        cv2.imwrite(f'{ruta}/{cont}.jpg', rostro_resized)
        cont += 1
        cv2.rectangle(frame, (x, y),
                      (x+w, y+h), (0, 255, 0), 2)

    cv2.imshow('Captura TF', frame)
    tecla = cv2.waitKey(1)
    if tecla == 27 or cont >= 70:
        break

cap.release()
cv2.destroyAllWindows()


entrenamiento de modelo

In [6]:
import os
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout)
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import ModelCheckpoint

# 1) Cargar datos
data = []
labels = []
dataset_path = 'dataset_tf'

for person in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person)
    if not os.path.isdir(person_path):
        continue
    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        data.append(img.astype('float32') / 255.0)
        labels.append(person)

data = np.expand_dims(np.array(data), -1)  # (N,100,100,1)

# 2) Codificar labels
le = LabelEncoder()
labels_enc = le.fit_transform(labels)
labels_cat = to_categorical(labels_enc)

# 3) Definir modelo
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(100,100,1)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(labels_cat.shape[1], activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4) Entrenar
checkpoint = ModelCheckpoint('recognizer_tf/model.h5',
                             save_best_only=True,
                             monitor='val_accuracy',
                             mode='max')
history = model.fit(data, labels_cat,
                    epochs=20,
                    batch_size=16,
                    validation_split=0.2,
                    callbacks=[checkpoint])

# 5) Guardar encoder
import pickle
os.makedirs('recognizer_tf', exist_ok=True)
with open('recognizer_tf/label_encoder.pickle', 'wb') as f:
    pickle.dump(le, f)

print("Entrenamiento completado y modelo guardado en recognizer_tf/")


Epoch 1/20


c:\Users\usser\anaconda3\envs\entorno_pdl\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5830 - loss: 0.6591

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.5994 - loss: 0.6548 - val_accuracy: 1.0000 - val_loss: 0.3795
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9240 - loss: 0.3728 - val_accuracy: 1.0000 - val_loss: 0.0070
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.0465 - val_accuracy: 1.0000 - val_loss: 0.0066
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9573 - loss: 0.0397 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9903 - loss: 0.0114 - val_accuracy: 1.0000 - val_loss: 1.2772e-08
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 1.0000 - loss: 5.7479e-05 - val_accuracy: 1.0000 - val_loss: 1.5136e-05
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.9903 - loss: 0.0122 - val_accuracy: 1.0000 - val_loss: 2.1287e-07
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 5.5338e-05 - val_accuracy: 1.0000 - val_

recpnocimiento en tiempo real

In [8]:
import cv2
import numpy as np
import pickle
from tensorflow.keras.models import load_model

# 1) Cargar modelo y encoder
model = load_model('recognizer_tf/model.h5')
with open('recognizer_tf/label_encoder.pickle', 'rb') as f:
    le = pickle.load(f)

cap = cv2.VideoCapture(0)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades +
                                     'haarcascade_frontalface_default.xml')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        rostro = gray[y:y+h, x:x+w]
        rostro_resized = cv2.resize(rostro, (100, 100)).astype('float32') / 255.0
        rostro_input = np.expand_dims(np.expand_dims(rostro_resized, 0), -1)  # (1,100,100,1)

        preds = model.predict(rostro_input)
        idx = np.argmax(preds)
        prob = preds[0][idx]

        if prob > 0.5:
            nombre = le.inverse_transform([idx])[0]
            etiqueta = f"{nombre} ({prob*100:.1f}%)"
        else:
            etiqueta = "Desconocido"

        cv2.putText(frame, etiqueta,
                    (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.rectangle(frame, (x, y),
                      (x+w, y+h), (0, 255, 0), 2)

    cv2.imshow('Reconocimiento TensorFlow', frame)
    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━